# Tutorial — Esteira de seleção de variáveis (`ModelSegmenter.select_features`)

Selecionar variáveis, na prática, sempre foi um punhado de pedaços soltos: um corte de faltantes aqui,
um ranking de **IV** ali, um **PSI** para ver estabilidade, uma matriz de correlação no fim. Cada pedaço
num trecho de código diferente — e, quando o comitê pergunta *"por que essa variável saiu?"*, a resposta
depende de alguém lembrar qual régua estava valendo naquele dia.

A **esteira de seleção** junta esses pedaços num fluxo único. Você escolhe **quais** etapas quer rodar e
em **qual ordem**, faz **uma chamada** e recebe, para cada candidata: a decisão (**selecionada** ·
**a revisar** · **excluída**), **em qual etapa** ela saiu e **por quê**, em texto apresentável — mais o
**funil** por etapa, os gráficos, um **relatório HTML autocontido** e a **política em JSON** para
reproduzir exatamente a mesma régua meses depois.

Ao final deste tutorial você vai conseguir:

- rodar a seleção inteira com **uma linha** (`seg.select_features()`) e ler o resultado;
- **escolher e ordenar** as etapas — e entender por que a ordem muda os números;
- tratar **categóricas** como categóricas (cardinalidade, categorias raras, faltante como faixa);
- **simular** uma régua sem tocar no segmentador, comparar e só então aplicar;
- gerar o **relatório** (HTML + Excel) que vai anexo à documentação do modelo;
- guardar a **política** e reaplicá-la em outra base/safra.

> **Contrato de dados:** um `pandas.DataFrame` com a coluna-alvo (`target`), as candidatas com prefixo
> `feat_` e — opcionais, mas usados aqui — a coluna de amostra (`amostra`: DES/OOT) e a de safra
> (`dt_ref`). A esteira mede tudo na **amostra de referência** (`ref_sample`) e compara as demais.
>
> **Rótulos neutros:** nada aqui é específico de um parâmetro de risco. O alvo é sempre nomeado pelo
> `problem_label` do segmentador — aqui, `"inadimplência 12m"`.

**Roteiro:** 0) setup · 1) base sintética · 2) o caso simples · 3) escolhendo as etapas ·
4) categóricas em detalhe · 5) simular sem aplicar · 6) os gráficos · 7) o relatório final ·
8) reprodutibilidade · 9) a mesma esteira na interface · 10) quando usar cada etapa.

## 0. Setup

A primeira célula só torna o pacote `yggdrasil` importável a partir do repositório (sem `pip install`) —
é a mesma dos demais tutoriais e é inócua se o pacote já estiver instalado. A segunda traz os imports e
`%matplotlib inline`, para os gráficos aparecerem no próprio notebook.

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path

def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None

_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
from IPython.display import display   # autossuficiente fora do kernel (nbconvert etc.)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)

## 1. A base do tutorial

A base é **sintética e autocontida** (nada é lido do disco): 3.000 contratos, 12 safras, amostra `DES`
(desenvolvimento) e `OOT` (as três últimas safras). Cada variável foi desenhada para cair — ou sobreviver
— numa etapa específica da esteira, de modo que o funil conte uma história completa:

| variável | o que é | papel esperado |
|---|---|---|
| `feat_score_bureau` | score de crédito do bureau (↑ = melhor) | **forte**, sobrevive |
| `feat_util_limite` | utilização do limite | média, sobrevive |
| `feat_qt_consultas_3m` | consultas ao CPF em 3 meses | fraca, mas acima do piso de IV |
| `feat_prazo_meses` | prazo contratado (risco alto nas **duas** pontas) | média, sobrevive |
| `feat_score_bureau_v2` | cópia ruidosa do score do bureau | **redundante** → sai na *correlação* |
| `feat_comprometimento` | comprometimento de renda, com **régua trocada na OOT** | **instável** → sai no *PSI* |
| `feat_ruido_cadastral` | campo sem relação com o alvo | **sem sinal** → sai no *IV* |
| `feat_renda_informada` | renda informada, preenchida em ~15% dos casos | **85% faltante** → sai nos *faltantes* |
| `feat_flag_legado` | campo legado sempre = 1 | **constante** → sai nas *constantes* |
| `feat_marcacao_cobranca` | marcação criada **depois** da decisão de crédito | **IV alto demais** → *a revisar* |
| `feat_agencia` | agência de origem (60 valores) | **alta cardinalidade** → sai nas *categóricas* |
| `feat_canal` | canal de originação (3 canais + 5 pilotos minúsculos, 8% faltante) | sobrevive **agrupada** |

> Os fatores latentes (`z_bureau`, `z_endiv`, `z_util`) geram o risco e **não** entram na base — só as
> variáveis observáveis viram colunas `feat_`.

In [ ]:
rng = np.random.default_rng(7)
n = 3000

# safras e amostras: as 3 últimas safras viram OOT (fora do tempo)
safras = pd.date_range("2024-01-01", periods=12, freq="MS")
dt_ref = rng.choice(safras, size=n)
amostra = np.where(dt_ref >= safras[9], "OOT", "DES")

# fatores latentes que geram o risco (não entram na base)
z_bureau = rng.normal(size=n)      # qualidade de crédito vista pelo bureau
z_endiv  = rng.normal(size=n)      # endividamento
z_util   = rng.normal(size=n)      # uso do limite

# canal de originação: 3 categorias frequentes + 5 pilotos minúsculos (~0,4% cada)
canal = np.array(rng.choice(["APP", "AGENCIA", "PARCEIRO"], size=n, p=[0.40, 0.35, 0.25]),
                 dtype=object)
for k, i in enumerate(rng.permutation(n)[:60]):
    canal[i] = f"PILOTO_{k % 5 + 1}"
efeito_canal = pd.Series(canal).map({"APP": -0.9, "AGENCIA": 0.0,
                                     "PARCEIRO": 0.9}).fillna(0.0).to_numpy()

# alvo binário: evento em 12 meses
lin = 0.6 * z_bureau + 0.5 * z_endiv + 0.4 * z_util + 0.6 * efeito_canal
y = rng.binomial(1, 1.0 / (1.0 + np.exp(-(lin - 1.0))))

df = pd.DataFrame({
    "target": y,
    "dt_ref": dt_ref,
    "amostra": amostra,
    # --- boas: sinal razoável e distribuição estável ----------------------
    "feat_score_bureau":    600 - 120 * z_bureau + rng.normal(0, 20, n),
    "feat_util_limite":     np.clip(0.35 + 0.12 * z_util + rng.normal(0, 0.06, n), 0, 1),
    "feat_qt_consultas_3m": rng.poisson(2 + np.clip(z_bureau, 0, None), n),
    "feat_prazo_meses":     np.clip(36 + np.where(y > 0, rng.choice([-10.0, 10.0], size=n), 0.0)
                                    + rng.normal(0, 10, n), 6, 72).round(),
    # --- redundante: cópia ruidosa do score do bureau ---------------------
    "feat_score_bureau_v2": np.nan,                      # preenchida logo abaixo
    # --- instável entre safras: a régua mudou na OOT ----------------------
    "feat_comprometimento": np.clip(0.30 + 0.12 * z_endiv + rng.normal(0, 0.04, n)
                                    + np.where(amostra == "OOT", 0.35, 0.0), 0, 2),
    # --- sem poder discriminante -----------------------------------------
    "feat_ruido_cadastral": rng.normal(size=n),
    # --- quase toda faltante ---------------------------------------------
    "feat_renda_informada": np.clip(3000 + 900 * rng.normal(size=n), 500, None),
    # --- constante (campo legado) ----------------------------------------
    "feat_flag_legado":     1.0,
    # --- boa demais: marcação criada DEPOIS da decisão (vazamento) --------
    "feat_marcacao_cobranca": 3.0 * y + rng.normal(0, 1.0, n),
    # --- categórica de alta cardinalidade (é chave, não preditora) --------
    "feat_agencia":         [f"AG{v:03d}" for v in rng.integers(0, 60, size=n)],
    # --- categórica com cauda rara + faltantes ----------------------------
    "feat_canal":           canal,
})
df["feat_score_bureau_v2"] = 0.98 * df["feat_score_bureau"] + rng.normal(0, 25, n)
df.loc[df.sample(frac=0.85, random_state=3).index, "feat_renda_informada"] = np.nan
df.loc[df.sample(frac=0.08, random_state=5).index, "feat_canal"] = None

print(f"{len(df)} linhas · {sum(c.startswith('feat_') for c in df.columns)} candidatas · "
      f"evento em {df['target'].mean():.1%} · OOT em {(df['amostra'] == 'OOT').mean():.1%}")
df.head()

## 2. O caso simples — **uma** chamada

Construa o segmentador com o contrato de sempre e chame `select_features()`. Sem argumento nenhum ele
roda a sequência default — `faltantes → constantes → categóricas → IV → PSI → monotonia → correlação` —
e **aplica** a decisão no segmentador (inclui/exclui, marca a categoria `manter`/`revisar`/`descartar` e
grava o motivo). É isto: uma linha para a seleção inteira.

`res.resumo()` é a leitura de bolso do que aconteceu: quantas entraram, quantas sobraram e o que cada
etapa levou embora.

In [ ]:
from yggdrasil.credit_risk.model import ModelSegmenter

seg = ModelSegmenter(df, target="target", task_type="classification",
                     sample_col="amostra", ref_sample="DES", date_col="dt_ref",
                     problem_label="inadimplência 12m", verbose=False)

res_padrao = seg.select_features()      # etapas default, já aplicadas no segmentador
print(res_padrao.resumo())

### 2.1. O funil

`res.funil` tem uma linha por etapa e **fecha aritmeticamente**: `Seguiram = Entraram − Excluídas`, e o
`Seguiram` de uma etapa é o `Entraram` da próxima. `A revisar` é sinalização — a variável segue no
conjunto, mas com ressalva registrada.

In [ ]:
res_padrao.funil

### 2.2. A tabela de decisões

Uma linha por candidata, na ordem de entrada: a decisão, a **etapa em que saiu** (vazio para quem
sobreviveu) e o **motivo por extenso**, já com os números que embasaram a conta. As métricas de quem caiu
*antes* da etapa que as calcula ficam vazias — a esteira não gasta binning com variável já descartada.

In [ ]:
res_padrao.tabela[["variavel", "tipo", "decisao", "etapa_saida", "iv", "forca",
                   "pior_psi", "missing_pct", "n_categorias", "motivo"]]

### 2.3. O que foi gravado no segmentador

Com `apply=True` (o default) a decisão **não fica só no relatório**: ela vira estado do segmentador —
a variável entra ou sai de `seg.included`, ganha a `categoria` (`manter`/`revisar`/`descartar`) e o
`motivo` no `var_meta` (o mesmo texto que a interface mostra no ranking). Daqui para a frente,
`seg.fit(...)` já treina só com as selecionadas.

In [ ]:
pd.DataFrame([{"variavel": f,
               "incluída": f in seg.included,
               "categoria": (seg.var_meta[f] or {}).get("categoria"),
               "motivo": (seg.var_meta[f] or {}).get("motivo")}
              for f in seg.candidates])

## 3. Escolhendo as etapas

`steps` é uma **lista** — as etapas rodam na ordem em que você as escreve, e cada uma recebe apenas as
**sobreviventes** da anterior. O catálogo abaixo sai do próprio registro da lib (`SELECTION_STEPS`), então
é sempre o que está de fato disponível.

In [ ]:
from yggdrasil.credit_risk.model.selection import (SELECTION_STEPS, STEPS_DEFAULT,
                                                   PARAMS_DEFAULT, etapas_disponiveis)

print("ordem canônica:", etapas_disponiveis())
pd.DataFrame([{"etapa": nome, "rótulo": step.rotulo,
               "no default?": "sim" if nome in STEPS_DEFAULT else "não",
               "o que faz": step.descricao}
              for nome, step in SELECTION_STEPS.items()])

As **réguas** são parâmetros nomeados de `select_features(**params)`, todos com default documentado em
`PARAMS_DEFAULT`. Os mais usados no dia a dia:

In [ ]:
principais = ["max_missing", "max_categorias", "min_freq_categoria", "agrupar_raras",
              "min_iv", "max_psi", "max_corr", "monotonia_exclui", "incluir_revisar"]
pd.DataFrame([{"parâmetro": p, "default": PARAMS_DEFAULT[p]} for p in principais])

# min_iv=None significa "piso da tarefa": 0,02 na classificação e 0,01 na regressão —
# na política ele já sai resolvido com o valor que de fato valeu (seção 8).

Um conjunto menor é perfeitamente válido. Exemplo: uma base **sem amostra de comparação** (só DES) não
tem PSI para medir — tire a etapa; e num primeiro corte exploratório talvez você queira só os filtros
duros mais o IV, deixando redundância para depois. Aqui usamos um segmentador auxiliar (`seg_demo`) e
`apply=False` para experimentar sem mexer na seleção que já aplicamos.

In [ ]:
seg_demo = ModelSegmenter(df, target="target", task_type="classification",
                          sample_col="amostra", ref_sample="DES", date_col="dt_ref",
                          problem_label="inadimplência 12m", verbose=False)

curta = seg_demo.select_features(steps=["missing", "constante", "categoricas", "iv"],
                                 apply=False)
print(curta.resumo())
curta.tabela[["variavel", "decisao", "etapa_saida", "motivo"]].head(12)

### 3.1. Por que a **ordem** importa

O default trata as **categóricas antes do IV** de propósito. O agrupamento de categorias raras muda a
binagem da variável — e o IV é calculado **em cima da binagem**. Medir o IV antes de agrupar dá um número
inflado por categorias minúsculas: cada piloto com 12 contratos vira uma faixa própria de risco quase
puro, e a variável parece muito mais forte do que é fora da amostra.

A lib não te impede de inverter a ordem, mas **avisa** — o alerta vai para o `warnings` e fica registrado
em `politica["avisos"]` (e no relatório).

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    invertida = seg_demo.select_features(steps=["iv", "categoricas"], apply=False)

print("aviso emitido:", avisos[0].message)
print()
print("registrado na política:", invertida.politica["avisos"])

### 3.2. `monotonia` **sinaliza**, não exclui

A etapa `monotonia` olha a ordem de risco entre as faixas: risco que sobe, desce e volta a subir é sinal
de faixa mal cortada (ou de relação em U). Por padrão isso vira **`revisar`**, não exclusão — a variável
pode ser ótima depois de reagrupada. Use `monotonia_exclui=True` quando a política do time exigir
monotonicidade estrita.

`feat_prazo_meses` é o caso clássico: risco alto nas **duas** pontas. Com o binning ótimo, o solver
encontra cortes que preservam a monotonia; a não-monotonia aparece quando o analista **fixa os cortes na
mão** — que é o caso realista de scorecard.

In [ ]:
seg_demo.set_manual_bins("feat_prazo_meses", [24.0, 48.0])   # cortes na mão: <24, 24–48, >48

sinaliza = seg_demo.select_features(steps=["monotonia"], apply=False,
                                    features=["feat_prazo_meses"])
exclui = seg_demo.select_features(steps=["monotonia"], apply=False,
                                  features=["feat_prazo_meses"], monotonia_exclui=True)

pd.concat([sinaliza.tabela.assign(regua="padrão"),
           exclui.tabela.assign(regua="monotonia_exclui=True")],
          ignore_index=True)[["regua", "variavel", "decisao", "tendencia",
                              "n_inversoes", "motivo"]]

## 4. Categóricas em detalhe

Categórica não é numérica com rótulo — e a etapa `categoricas` trata cada patologia com o remédio certo:

1. **cardinalidade alta** (acima de `max_categorias`, default 30): a variável é **chave**, não preditora.
   `feat_agencia`, com 60 agências, sai aqui — cada agência viraria uma faixa com pouquíssimos contratos,
   e o modelo decoraria a amostra em vez de aprender.
2. **categorias raras** (abaixo de `min_freq_categoria`, default 1%): são **agrupadas de fato** numa
   faixa "outros", usando os *bins manuais* do segmentador. O agrupamento vale para toda a análise
   univariada (tabela, WoE, IV, PSI) — e é por isso que ele precisa vir **antes** do IV.
3. **faltante**: vira **faixa própria** `(faltante)` no binning. Em crédito, "não informado" costuma ser
   informação (canal que não registra, cliente que não preenche) — jogar para a moda destruiria isso.

Repare no motivo de `feat_canal`: ele conta as três coisas de uma vez.

In [ ]:
cats = res_padrao.tabela[res_padrao.tabela["tipo"] == "cat"]
cats[["variavel", "decisao", "etapa_saida", "n_categorias", "missing_pct", "iv", "motivo"]]

O agrupamento fica **aplicado** no segmentador (é um bin manual), não é só uma nota no relatório:

In [ ]:
print("agrupamento aplicado em feat_canal:")
for grupo in seg.manual_bins("feat_canal"):
    print("  ", grupo)

# frequência das categorias na amostra de referência — os 5 pilotos somam menos de 2%
freq = (df.loc[df["amostra"] == "DES", "feat_canal"]
        .value_counts(normalize=True, dropna=False).rename("frequência"))
freq.to_frame()

As duas réguas são independentes. Afrouxando `max_categorias` a agência entra (e aí você vê o IV dela);
desligando `agrupar_raras` os pilotos continuam como faixas próprias — a esteira registra a ressalva no
motivo em vez de agrupar. Compare (tudo em simulação, sem aplicar):

In [ ]:
solta = seg_demo.select_features(steps=["categoricas", "iv"], apply=False,
                                 max_categorias=80, agrupar_raras=False,
                                 features=["feat_agencia", "feat_canal"])
solta.tabela[["variavel", "decisao", "n_categorias", "iv", "forca", "motivo"]]

### 4.1. Categórica nominal é **isenta** de monotonia

Cobrar monotonia de uma categórica nominal é erro conceitual: a ordem das faixas (`APP`, `AGENCIA`,
`PARCEIRO`, `OUTROS`) é **arbitrária** — trocar a ordem trocaria a "tendência" sem mudar nada na
variável. Por isso a etapa `monotonia` deixa as categóricas passarem explicitamente, com o motivo
registrado no histórico (e o `feat_canal` sobrevive mesmo com `tendencia = não-monotônica` na tabela).

In [ ]:
[h for h in res_padrao.historico
 if h["etapa"] == "monotonia" and h["variavel"] in ("feat_canal", "feat_prazo_meses")]

## 5. Simular sem aplicar (`apply=False`)

Antes de trocar a régua do modelo, vale ver o que a régua nova faria. Com `apply=False` a esteira roda
inteira, devolve a trilha completa — e **desfaz tudo** no fim: seleção, categorias e até os agrupamentos
de categorias raras voltam exatamente como estavam. É a forma segura de comparar réguas.

Aqui simulamos uma régua **mais apertada** (IV mínimo 0,10; PSI máximo 0,10; correlação máxima 0,70) e
comparamos decisão a decisão com a rodada default.

In [ ]:
antes = set(seg.included)

apertada = seg.select_features(apply=False, min_iv=0.10, max_psi=0.10, max_corr=0.70)

print(apertada.resumo())
print()
print("o segmentador mudou?", set(seg.included) != antes)   # False — nada foi aplicado

In [ ]:
comparacao = (
    res_padrao.tabela.set_index("variavel")[["decisao", "etapa_saida"]]
    .rename(columns={"decisao": "decisão (default)", "etapa_saida": "saiu em (default)"})
    .join(apertada.tabela.set_index("variavel")[["decisao", "etapa_saida"]]
          .rename(columns={"decisao": "decisão (apertada)", "etapa_saida": "saiu em (apertada)"}))
)
comparacao

Gostou da régua apertada? Rode de novo **sem** `apply=False` para valer. Note que `seg.selection_` guarda
sempre a **última** execução — é ela que o relatório e os gráficos usam quando você não passa
`result=...` explicitamente.

> A etapa `categoricas` reconhece o agrupamento que já está aplicado e o **preserva** (o motivo passa a
> dizer "agrupamento manual já definido — preservado"). Rodar a esteira duas vezes não empilha
> agrupamentos.

In [ ]:
res_final = seg.select_features(min_iv=0.10, max_psi=0.10, max_corr=0.70)

print(res_final.resumo())
print()
print("variáveis que vão ao modelo:", seg.selected_features())

### 5.1. E a coluna **a revisar**?

`feat_marcacao_cobranca` continua na lista: por padrão (`incluir_revisar=True`) quem sobreviveu **com
ressalva** segue incluída, marcada como `revisar`. Isso é proposital — a esteira nunca exclui sozinha por
IV alto demais, porque o diagnóstico é sobre a **origem** da variável: só quem conhece o dado sabe se a
marcação foi criada antes ou depois da decisão de crédito. O papel da etapa é levantar a mão.

Confirmado o vazamento, há duas saídas: tirar a variável na mão (`seg.exclude(...)`) ou rodar a esteira
com `incluir_revisar=False`, que deixa **todas** as ressalvadas fora do modelo.

In [ ]:
seg.exclude("feat_marcacao_cobranca")   # confirmado: a marcação nasce depois da decisão
print("variáveis que vão ao modelo:", seg.selected_features())

## 6. Os gráficos, um a um

Quatro figuras, pensadas como uma narrativa de apresentação. Todas leem a última seleção
(`seg.selection_`) ou o `result=` que você passar, e aceitam `save_path=` para gravar em PNG.

### 6.1. Funil — *de onde partimos e o que cada régua custou?*

In [ ]:
seg.plot_selection_funil()

### 6.2. Causas — *por que perdemos variáveis?* (a leitura executiva do funil)

O funil diz **quantas** saíram em cada etapa; este diz **por quê**, agrupando pela causa curta. Use
`por="motivo"` para abrir no texto por extenso, com os números de cada caso.

In [ ]:
seg.plot_selection_motivos()

### 6.3. Ranking de IV — *quem tem sinal, e onde ficou o corte?*

Barras coloridas pela decisão, com a linha tracejada no `min_iv` efetivamente usado — o corte sai da
política, não de um número digitado no gráfico.

In [ ]:
seg.plot_selection_iv(top=12)

### 6.4. IV × PSI — *o sinal é forte **e** estável?*

A matriz de decisão: os cortes de IV e de PSI dividem o plano em quatro leituras. O canto inferior
direito (*forte e estável*) é o território desejável; *forte mas instável* é a variável que engana na
amostra de desenvolvimento e quebra na OOT.

In [ ]:
seg.plot_selection_iv_psi(annotate_top=6)

## 7. O relatório final — HTML e Excel

O HTML é uma página **autocontida**: CSS embutido, figuras em `data:image/png;base64` e nenhuma
requisição externa. Dá para anexar ao material do comitê, mandar por e-mail ou guardar como evidência de
como a lista de variáveis foi construída — o arquivo abre sozinho no navegador, sem o notebook.

Ele traz, na ordem: sumário executivo em cartões, as principais causas de exclusão, o funil (gráfico +
tabela), as causas, o ranking de IV, o IV × PSI, a tabela completa de decisões com o motivo por extenso e
a **política** usada.

Gravamos num diretório temporário para não sujar o repositório — na sua análise, troque por um caminho
persistente (no Databricks, DBFS ou Volumes).

In [ ]:
import tempfile
from pathlib import Path

pasta = Path(tempfile.mkdtemp(prefix="selecao_"))

caminho_html = seg.selection_report(str(pasta / "selecao_inadimplencia_12m.html"),
                                    title="Seleção de variáveis · inadimplência 12m",
                                    subtitle="base sintética do tutorial · referência DES")
print("relatório salvo em:", caminho_html)
print("tamanho:", f"{Path(caminho_html).stat().st_size / 1024:,.0f} KB")

# sem `path`, o método devolve o HTML como string — dá para exibir no próprio notebook
# com IPython.display.HTML(...), embora a página tenha CSS próprio de documento inteiro.
html = seg.selection_report()
print("prévia:", html[:120], "...")

O Excel multi-abas (**Decisoes · Funil · Politica**) é para quem vai continuar a análise na planilha:
os valores vêm **numéricos** (IV, PSI, faltantes como percentual de verdade), não formatados como texto.

`openpyxl` é uma dependência **opcional** — a biblioteca não a exige para nada além disto, e o erro,
quando ela falta, é explícito.

In [ ]:
try:
    caminho_xlsx = seg.selection_xlsx(str(pasta / "selecao_inadimplencia_12m.xlsx"))
    print("Excel salvo em:", caminho_xlsx)
except ImportError as e:
    print("Excel não gerado —", e)

## 8. Reprodutibilidade — a política em JSON

`seg.selection_policy_` guarda a **política efetiva** da última execução: as etapas na ordem em que
rodaram e **todos** os parâmetros já resolvidos — inclusive os que você não digitou (o `min_iv` default,
por exemplo, é 0,02 na classificação e 0,01 na regressão; na política ele aparece com o valor que
realmente valeu). Nada fica implícito, e o dicionário é 100% serializável em JSON.

In [ ]:
import json

politica = seg.selection_policy_
print(json.dumps({k: v for k, v in politica.items() if k != "parametros"},
                 indent=2, ensure_ascii=False))
print()
print("parâmetros efetivos:")
print(json.dumps(politica["parametros"], indent=2, ensure_ascii=False))

Como os parâmetros já vêm resolvidos, **reaplicar a mesma régua** em outra base (a safra seguinte, outra
carteira, um recorte) é passar a política de volta:

In [ ]:
seg_nova_safra = ModelSegmenter(df, target="target", task_type="classification",
                                sample_col="amostra", ref_sample="DES", date_col="dt_ref",
                                problem_label="inadimplência 12m", verbose=False)

res_reaplicado = seg_nova_safra.select_features(steps=politica["etapas"],
                                                **politica["parametros"])

print(res_reaplicado.resumo())
print()
print("mesma lista da rodada original?",
      res_reaplicado.selecionadas == res_final.selecionadas)

A política também viaja com o modelo: ela entra no `to_dict()`/`save()` na chave `selection_policy` e
volta no `from_dict()`/`load()` (JSON antigo, sem a chave, devolve `None`). A **trilha completa**
(`seg.selection_`) é da sessão que rodou — o que persiste é a régua, que é o que se reproduz.

In [ ]:
config = seg.to_dict()
print("chaves da política salva:", sorted(config["selection_policy"]))
print("etapas gravadas no modelo:", config["selection_policy"]["etapas"])

## 9. A mesma esteira na interface

Na `ModelSegmenterUI`, o card **Esteira de seleção** (aba ① *Variáveis*) é a mesma coisa com caixinhas:
uma por etapa (na ordem canônica), os campos das réguas, o botão **Rodar seleção**, a caixa **apenas
simular** e os dois botões de exportação. A interface **não tem régua própria** — ela chama
`select_features` com os mesmos defaults.

Como um notebook estático não registra cliques, a célula seguinte dirige o card por código: marca as
etapas, ajusta uma régua, liga a simulação e chama o *handler* do botão.

In [ ]:
from yggdrasil.credit_risk.model import ModelSegmenterUI

ui = ModelSegmenterUI(df, target="target", task_type="classification",
                      problem_label="inadimplência 12m", sample_col="amostra",
                      ref_sample="DES", date_col="dt_ref")
ui   # exibe o workbench; o card "Esteira de seleção" fica na aba ① Variáveis

In [ ]:
# marcar/desmarcar etapas = clicar nas caixinhas do card
for nome, caixa in ui._sel_step_cbs.items():
    caixa.value = nome in ("missing", "constante", "categoricas", "iv")

ui.fl_sel_min_iv.value = 0.05        # campo "IV mín."
ui.cb_sel_simular.value = True       # "apenas simular (não aplica)"
ui._on_selection_run(None)           # = clicar em "Rodar seleção"

print(ui.seg.selection_.resumo())    # o resultado da esteira rodada pela UI

## 10. Quando usar cada etapa

| etapa | use quando | cuidado |
|---|---|---|
| `missing` | sempre — é o filtro mais barato | limite alto demais deixa passar variável que só existe numa safra |
| `constante` | sempre | pega também a "quase-constante" (categoria dominante acima de `max_dominancia`) |
| `categoricas` | há categóricas na base | **antes** do IV, sempre |
| `iv` | sempre que houver alvo | IV altíssimo é *revisar*, não aprovação: quase sempre é vazamento |
| `psi` | há mais de uma amostra (DES/OOT) | sem amostra de comparação a etapa não avalia ninguém e avisa |
| `monotonia` | scorecard / modelo que precisa de leitura de negócio | por padrão **sinaliza**; só exclui com `monotonia_exclui=True` |
| `correlacao` | sempre, no fim | precisa do IV para escolher quem fica no par — por isso vem **depois** do IV |

**O que fica de fora do default, e por quê:**

- **`vif`** — mede multicolinearidade na *matriz de desenho do modelo vigente*, então exige um modelo já
  ajustado (`seg.fit(...)`); sem isso a esteira levanta erro explicando. Vale quando o modelo é linear e
  você precisa defender os coeficientes um a um. Por padrão também **sinaliza** (`vif_exclui=True` exclui).
- **`backward`** — treina **dezenas de modelos** (remove a variável menos importante e re-treina, passo a
  passo). É a etapa mais cara da lista e faz sentido no fim do ciclo, quando a lista já está limpa e a
  pergunta é "de quantas variáveis o modelo realmente precisa?".

Ambas são etapas normais: basta incluí-las em `steps` (ex.:
`seg.select_features(steps=[*STEPS_DEFAULT, "backward"])`).

**Fecho.** A esteira é um **ponto de partida defensável**, não um oráculo: ela deixa registrado o que foi
medido, com qual régua e por que cada variável saiu. A validação final da lista continua sendo do time de
modelagem — só que agora com a trilha inteira anexada.

---

Outros tutoriais (mesma pasta): `00_tutorial_yggdrasil`, `01_tutorial_lgd`, `02_tutorial_eda_features`,
`03_tutorial_feature_selection`, `04_tutorial_tree_segmenter`, `05_tutorial_instalacao_e_interfaces`,
`06_tutorial_model_segmenter`, `07_tutorial_esteira_ml_mlflow`, `08_tutorial_capital_economico`,
`09_tutorial_modelos_econometricos`.